In [24]:
# --- Modelo Preditivo. Baseline simples, utilizando média móvel: 
# O modelo utiliza a média móvel para prever os próximos valores assumindo que o valor continuará constante ---
# Objetivo: Prever o tempo necessário para repor estoque de um certo produto, através da média móvel.
# Período selecionado para calcular a média: Última semana de 2023 (25-12-2023 à 31-12-2023).
# Período para aplicar a previsão: Janeiro de 2024.
# O arquivo segue uma organização de arquivos dia_semana
# /datasets/vendas_2023_2024.csv

import pandas as pd

# Criação do range do calendário
datas = pd.date_range(start='2023-01-01', end='2024-12-31')
df_calendario = pd.DataFrame(datas, columns=['data'])

# Formatação para dias das semana e meses serem em pt-BR
meses_pt = {
    'January':'Janeiro','February':'Fevereiro','March':'Marco','April':'Abril','May':'Maio',
    'June':'Junho','July':'Julho','August':'Agosto','September':'Setembro','October':'Outubro',
    'November':'Novembro','December':'Dezembro'
}

dias_pt = {
    'Monday':'Segunda-feira','Tuesday':'Terca-feira','Wednesday':'Quarta-feira',
    'Thursday':'Quinta-feira','Friday':'Sexta-feira','Saturday':'Sabado','Sunday':'Domingo'
}

# Extração das infos('ano', 'mes', 'nome_mes', 'dia_semana', 'data') para montar o Dataframe
df_calendario['ano'] = df_calendario['data'].dt.year
df_calendario['mes'] = df_calendario['data'].dt.month
df_calendario['nome_mes'] = df_calendario['data'].dt.month_name().map(meses_pt)
df_calendario['dia_semana'] = df_calendario['data'].dt.day_name().map(dias_pt)
df_calendario['data'] = df_calendario['data'].dt.date

# Transformação do Dataframe para .csv
df_calendario.to_csv('dia_semana.csv', index=False)

In [25]:
# Dados da tabela de vendas
df_vendas = pd.read_csv("datasets/vendas_2023_2024.csv")
# Calendário com range 01-01-2023 à 31-12-2024
df_dias = pd.read_csv('dia_semana.csv')

In [26]:
# Renomeação da coluna data do Dataframe de produtos
df_dias.rename(columns={'data':'sale_date'}, inplace=True)

In [27]:
# Junção das tabelas por left join pelo calendário, para encontrar possíveis valores nulos
df_final = pd.merge(df_dias,df_vendas,on='sale_date', how='left')
# Exibição dos valores nulos
df_final.isnull().sum()

sale_date     0
ano           0
mes           0
nome_mes      0
dia_semana    0
id            8
id_client     8
id_product    8
qtd           8
total         8
dtype: int64

In [28]:
# Reatribuição e tratamento dos dados
df_final = df_final.fillna(0)
# Checagem dos valores nulos após tratamento
df_final.isnull().sum()

sale_date     0
ano           0
mes           0
nome_mes      0
dia_semana    0
id            0
id_client     0
id_product    0
qtd           0
total         0
dtype: int64

In [29]:
# Agrupamento de dados por sale_date e id_product, somando as qtd de produtos dos dados agrupados, reset_index para formatar a tabela novamente
df_diario = df_final.groupby(['sale_date', 'id_product'])['qtd'].sum().reset_index()

In [30]:
# Determinando limite de dados a ser analisado
ultima_semana_2023 = df_diario[
    (df_diario['sale_date'] >= '2023-12-25') & 
    (df_diario['sale_date'] < '2024-01-01')
]

In [31]:
# Cálculo do baseline através da média móvel de qtd por produtos, utilizando a última semana de 2023 como referência
baseline_produtos = ultima_semana_2023.groupby('id_product')['qtd'].mean().reset_index()
# Renomeação da coluna qtd para média diária
baseline_produtos.rename(columns={'qtd':'media_diaria'}, inplace=True)

In [32]:
# Criação do Dataframe para receber as datas de Janeiro de 2024
periodo_janeiro = pd.date_range(start='2024-01-01', end='2024-01-31')
# Alimentação para o Dataframe através das sale_dates para o Dataframe df_calendario_janeiro
df_calendario_janeiro = pd.DataFrame({'sale_date':periodo_janeiro})

In [33]:
# Criação de uma coluna['key] = 1 para juntar os dois Dataframes
df_calendario_janeiro['key'] = 1
baseline_produtos['key'] = 1

In [34]:
# Junção dos Dataframes e dropando a coluna key, pois não será mais necessário
previsao_diaria = pd.merge(df_calendario_janeiro, baseline_produtos, on='key').drop('key', axis=1)

In [40]:
# Formatação para a média diária se tornar um valor inteiro, não é possível vender 1,5 de algum produto
previsao_diaria['previsao_venda_diaria'] = previsao_diaria['media_diaria'].round(0)
previsao_diaria

,sale_date,id_product,media_diaria,previsao_venda_diaria
0,2024-01-01,3.0,10.0,10.0
1,2024-01-01,7.0,10.0,10.0
2,2024-01-01,10.0,10.0,10.0
3,2024-01-01,11.0,19.0,19.0
4,2024-01-01,13.0,3.0,3.0
...,...,...,...,...
1514,2024-01-31,133.0,9.0,9.0
1515,2024-01-31,141.0,15.0,15.0
1516,2024-01-31,145.0,2.0,2.0
1517,2024-01-31,148.0,12.0,12.0


In [ ]:
# Leitura do arquivo produtos e renomeação da tabela code para id_product
df_produtos = pd.read_csv('concluidos/produtos_raw_tratados.csv')
df_produtos.rename(columns={'code':'id_product'},inplace=True)

In [41]:
# Junção dos Dataframes com a média prevista e produtos
df_final = pd.merge(previsao_diaria,df_produtos,on='id_product',how='left')
df_final

,sale_date,id_product,media_diaria,previsao_venda_diaria,name,price,actual_category
0,2024-01-01,3.0,10.0,10.0,Radar Furuno Pulse Leviathan,9024.19,eletrônicos
1,2024-01-01,7.0,10.0,10.0,Radar AIS Zen,19518.77,eletrônicos
2,2024-01-01,10.0,10.0,10.0,Piloto Automático Simrad Titan Flux Magnum,32033.04,eletrônicos
3,2024-01-01,11.0,19.0,19.0,GPS Furuno Swift Leviathan Poseidon,36231.91,eletrônicos
4,2024-01-01,13.0,3.0,3.0,Radar Simrad Boost,7747.10,eletrônicos
...,...,...,...,...,...,...,...
1514,2024-01-31,133.0,9.0,9.0,Âncora Bruce Core Pulse,1892.72,ancoragem
1515,2024-01-31,141.0,15.0,15.0,Boia de Arqueamento Bruce Nexus Abyss,4781.80,ancoragem
1516,2024-01-31,145.0,2.0,2.0,Boia de Arqueamento Delta Nexus,4349.86,ancoragem
1517,2024-01-31,148.0,12.0,12.0,Âncora Delta Force Barracuda Mako,4785.56,ancoragem


In [ ]:
# Definição de colunas a serem visualizadas na consulta e exibição
colunas_finais = ['sale_date', 'id_product', 'name', 'previsao_venda_diaria']
df_relatorio = df_final[colunas_finais]

df_relatorio.to_csv('Relatorio_baseline_para_compra_estoque.csv', index=False)

In [ ]:
# Definição para analisar um produto em específico e delimitando a primeira semana de 2024
venda_semanal_motor155hp = df_relatorio[
    (df_relatorio['id_product'] == 54.0) & 
    (df_relatorio['sale_date'] >= '2024-01-01') & 
    (df_relatorio['sale_date'] <= '2024-01-07')
]
# Atribuição e visualização da previsão total para o Motor 155 HP
total_previsao = venda_semanal_motor155hp['previsao_venda_diaria'].sum()
print(f"Previsão total para semana de janeiro: {int(total_previsao)}")

Previsão total para semana de janeiro: 0
